In [1]:
import tensorflow as tf
tf.keras.backend.clear_session()


In [2]:
import numpy as np
from torch.cuda.amp import autocast, GradScaler
import pandas as pd
import os
import cv2
import math
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import cv2
import os
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset
from torchvision.models.vision_transformer import vit_b_16
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import torch
import torch.nn as nn
from torch.nn.init import trunc_normal_
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import cv2
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm

In [3]:
import tensorflow as tf

# Check if GPU is available
if tf.config.list_physical_devices('GPU'):
    print("Using GPU")
else:
    print("Using CPU")

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Using GPU
Num GPUs Available:  2


In [4]:
def drop_path(x, drop_prob: float = 0., training: bool = False):
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    output = x.div(keep_prob) * random_tensor
    return output

In [5]:
class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

class Attention(nn.Module):
    def __init__(self, dim, hidden_dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.head_dim = head_dim
        self.scale = head_dim ** -0.5

        self.qk = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.v = nn.Linear(dim, dim, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop, inplace=False)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop, inplace=True)

    def forward(self, x, relative_pos=None):
        B, N, C = x.shape
        qk = self.qk(x).reshape(B, N, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k = qk.unbind(0)
        v = self.v(x).reshape(B, N, self.num_heads, -1).permute(0, 2, 1, 3)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        if relative_pos is not None:
            attn += relative_pos
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, -1)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


In [6]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        drop_probs = (drop, drop)

        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.drop1 = nn.Dropout(drop_probs[0])
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop2 = nn.Dropout(drop_probs[1])

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x

In [7]:
class Merge(nn.Module):
    def __init__(self, in_dim, out_dim, patch_size, in_chans=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_dim*2, in_dim*4, kernel_size=1, stride=1),
            nn.BatchNorm2d(in_dim*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_dim*4, in_dim*4, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(in_dim*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_dim*4, out_dim, kernel_size=1, stride=1),
            nn.BatchNorm2d(out_dim),
            nn.ReLU(inplace=True),
            )
        self.in_dim = in_dim
        self.patch_size = patch_size
        self.norm_in = nn.LayerNorm(in_dim)
        self.norm_out = nn.LayerNorm(out_dim)
    
    def forward(self, pixel_embed1, pixel_embed2):
        H_p = W_p = self.patch_size
        W_column = pixel_embed1.size(1)
        BW_column, H_row, _ = pixel_embed2.size()
        B = BW_column // W_column
        assert H_row == W_column
    
        img1 = pixel_embed1.reshape(B, H_row, W_column, H_p, W_p, self.in_dim).permute(0, 5, 1, 3, 2, 4).reshape(B, self.in_dim, H_row*H_p, W_column*W_p)
        img2 = pixel_embed2.reshape(B, H_row, W_column, H_p, W_p, self.in_dim).permute(0, 5, 1, 3, 2, 4).reshape(B, self.in_dim, H_row*H_p, W_column*W_p)
        img_reshaped = torch.cat([img1, img2], dim=1)
        img_merge = self.conv(img_reshaped)

        return img_merge

In [8]:
class Block(nn.Module):
    def __init__(self, dim, words_in_sentence, patch_size, sentences, in_chans=3, num_heads=2, num_inner_heads=4, mlp_ratio=4.,
            qkv_bias=False, drop=0., attn_drop=0., drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.patch_size = patch_size
        words = patch_size * patch_size
        self.norm_in = norm_layer(dim*words)
        self.attn_in1 = Attention(
            dim*words, dim*words, num_heads=num_inner_heads, qkv_bias=qkv_bias,
            attn_drop=attn_drop, proj_drop=drop)
        self.attn_in2 = Attention(
            dim*words, dim*words, num_heads=num_inner_heads, qkv_bias=qkv_bias,
            attn_drop=attn_drop, proj_drop=drop)
        self.norm_mlp_in = norm_layer(dim*words)
        self.mlp_in1 = Mlp(in_features=dim*words, hidden_features=int(dim*words * 4),
            out_features=dim*words, act_layer=act_layer, drop=drop)
        self.mlp_in2 = Mlp(in_features=dim*words, hidden_features=int(dim*words * 4),
            out_features=dim*words, act_layer=act_layer, drop=drop)
        
        self.norm_proj = norm_layer(dim*words)
        self.proj1 = nn.Linear(dim*words, dim*words, bias=True)
        self.proj2 = nn.Linear(dim*words, dim*words, bias=True)

        self.norm_out = norm_layer(dim * words_in_sentence)
        self.attn_out1 = Attention(
            dim * words_in_sentence, dim * words_in_sentence, num_heads=num_heads, qkv_bias=qkv_bias,
            attn_drop=attn_drop, proj_drop=drop)
        self.attn_out2 = Attention(
            dim * words_in_sentence, dim * words_in_sentence, num_heads=num_heads, qkv_bias=qkv_bias,
            attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        
        self.norm_mlp = norm_layer(dim * words_in_sentence)
        self.mlp1 = Mlp(in_features=dim * words_in_sentence, hidden_features=int(dim * words_in_sentence * mlp_ratio),
            out_features=dim * words_in_sentence, act_layer=act_layer, drop=drop)
        self.mlp2 = Mlp(in_features=dim * words_in_sentence, hidden_features=int(dim * words_in_sentence * mlp_ratio),
            out_features=dim * words_in_sentence, act_layer=act_layer, drop=drop)

    def forward(self, pixel_embed1, pixel_embed2, row_embed, column_embed, relative_pos=None):
        _, W_grid, _ = pixel_embed1.size()
        H_grid = W_grid
        H_p = W_p = self.patch_size
        B, N, C = row_embed.size()
        
        assert N == H_grid
        row_embed = row_embed + self.drop_path(self.attn_out1(self.norm_out(row_embed)))
        row_embed = row_embed + self.drop_path(self.mlp1(self.norm_mlp(row_embed)))

        assert N == W_grid
        column_embed = column_embed + self.drop_path(self.attn_out2(self.norm_out(column_embed)))
        column_embed = column_embed + self.drop_path(self.mlp2(self.norm_mlp(column_embed)))

        pixel_embed1 = pixel_embed1 + self.proj1(self.norm_proj(row_embed.reshape(B*H_grid, H_p, W_grid, W_p, -1).transpose(1, 2).reshape(B*H_grid, W_grid, -1)))
        attn_patch1 = self.attn_in1(self.norm_in(pixel_embed1.reshape(B, H_grid*W_grid, -1)))
        pixel_embed1 = pixel_embed1 + self.drop_path(attn_patch1.reshape(B*H_grid, W_grid, -1))
        pixel_embed1 = pixel_embed1 + self.proj2(self.norm_proj(column_embed.reshape(B, W_grid, H_grid, -1).transpose(1, 2).reshape(B*H_grid, W_grid, -1)))
        attn_patch2 = self.attn_in2(self.norm_in(pixel_embed1.reshape(B, H_grid*W_grid, -1)))
        pixel_embed1 = pixel_embed1 + self.drop_path(attn_patch2.reshape(B*H_grid, W_grid, -1))
        pixel_embed1 = pixel_embed1 + self.drop_path(self.mlp_in1(self.norm_mlp_in(pixel_embed1)))

        pixel_embed2 = pixel_embed2 + self.proj2(self.norm_proj(column_embed.reshape(B, W_grid, H_grid, -1).transpose(1, 2).reshape(B*H_grid, W_grid, -1)))
        attn_patch3 = self.attn_in2(self.norm_in(pixel_embed2.reshape(B, H_grid*W_grid, -1)))
        pixel_embed2 = pixel_embed2 + self.drop_path(attn_patch3.reshape(B*H_grid, W_grid, -1))
        pixel_embed2 = pixel_embed2 + self.proj1(self.norm_proj(row_embed.reshape(B*H_grid, H_p, W_grid, W_p, -1).transpose(1, 2).reshape(B*H_grid, W_grid, -1)))
        attn_patch4 = self.attn_in1(self.norm_in(pixel_embed2.reshape(B, H_grid*W_grid, -1)))
        pixel_embed2 = pixel_embed2 + self.drop_path(attn_patch4.reshape(B*H_grid, W_grid, -1))
        pixel_embed2 = pixel_embed2 + self.drop_path(self.mlp_in2(self.norm_mlp_in(pixel_embed2)))

        return pixel_embed1, pixel_embed2, row_embed, column_embed


In [9]:
class ToEmbed(nn.Module):
    def __init__(self, img_size=256, in_chans=3, patch_size=2, dim=8):
        super().__init__()
        img_size_tuple = (img_size, img_size)
        row_patch_size = (patch_size, img_size)
        self.grid_size = (img_size_tuple[0] // row_patch_size[0], img_size_tuple[1] // row_patch_size[1])
        num_patches = self.grid_size[0] * self.grid_size[1]
        self.patch_size = patch_size
        self.img_size = img_size_tuple
        self.num_patches = num_patches
        self.row_patch_size = row_patch_size
        self.dim = dim
        row_pixel = row_patch_size[0] * row_patch_size[1]

        self.unfold = nn.Unfold(kernel_size=patch_size, stride=patch_size)
        self.norm_proj = nn.LayerNorm(row_pixel * dim)
        self.proj1 = nn.Linear(row_pixel * dim, row_pixel * dim)
        self.proj2 = nn.Linear(row_pixel * dim, row_pixel * dim)

    def forward(self, x, pixel_pos=None):
        B, C, H, W = x.shape
        assert H == self.img_size[0]
        assert W == self.img_size[1]

        x = self.unfold(x)
        if pixel_pos is not None:
            x = x + pixel_pos
        x = x.transpose(1, 2).reshape(B , self.num_patches, self.num_patches, self.dim, self.patch_size, self.patch_size)
        
        pixel_embed = x.permute(0, 1, 2, 4, 5, 3).reshape(B * self.num_patches, -1, self.patch_size*self.patch_size*self.dim)
        row_embed = self.norm_proj(x.permute(0, 1, 4, 2, 5, 3).reshape(B, self.num_patches, -1))
        column_embed =  self.norm_proj(x.permute(0, 2, 1, 4, 5, 3).reshape(B, self.num_patches, -1))
        
        return pixel_embed, pixel_embed, row_embed, column_embed

In [10]:
class Stage(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, dim, out_dim, num_heads=2, num_inner_head=2, depth=1, 
                     mlp_ratio=4., qkv_bias=False, drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1, 
                     norm_layer=nn.LayerNorm):
        super().__init__()

        self.pixel_embed = ToEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans, dim=dim)
        row_patch_size = self.pixel_embed.row_patch_size        
        self.row_pixel = row_patch_size[0] * row_patch_size[1]
        self.patch_pixel = patch_size*patch_size
        self.num_patches = self.pixel_embed.num_patches

        blocks = []
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        for i in range(depth):
            blocks.append(Block(
                dim=dim, words_in_sentence=self.row_pixel, num_heads=num_heads, num_inner_heads=num_inner_head,
                mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, drop=drop_rate, attn_drop=attn_drop_rate, in_chans=in_chans,
                drop_path=dpr[i], norm_layer=norm_layer, patch_size=patch_size, sentences=self.num_patches))
        self.blocks = nn.ModuleList(blocks)
        self.merge = Merge(in_dim=dim, out_dim=out_dim, patch_size=patch_size, in_chans=in_chans,)

        self.pos_drop = nn.Dropout(p=drop_rate)

    def forward(self, x, pixel_pos=None, row_pos=None, column_pos=None):
        pixel_embed1, pixel_embed2, row_embed, column_embed = self.pixel_embed(x, pixel_pos)
        if row_pos is not None:
            row_embed = row_embed + row_pos
        row_embed = self.pos_drop(row_embed)
        if column_pos is not None:
            column_embed = column_embed + column_pos
        column_embed = self.pos_drop(column_embed)
        for blk in self.blocks:
            pixel_embed1, pixel_embed2, row_embed, column_embed = blk(pixel_embed1, pixel_embed2, row_embed, column_embed)
        img_merge = self.merge(pixel_embed1, pixel_embed2)
        
        return img_merge

In [11]:
class HoverTrans(nn.Module):
    def __init__(self, img_size=128, patch_size=[32, 2, 2, 2], in_chans=3, num_classes=5, embed_dim=768, 
                 dim=[48, 96, 192, 384], depth=[1, 2, 6, 2], num_heads=[2, 4, 8, 16], num_inner_head=[4, 4, 4, 4], 
                 mlp_ratio=4., drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1, norm_layer=nn.LayerNorm):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim 

        stride = [4, 2, 2, 2]
        self.stage = nn.ModuleList([])
        self.downsample = nn.ModuleList([])
        
        for i in range(4):
            if i == 0:
                self.stage.append(Stage(img_size=img_size//stride[i], patch_size=patch_size[i], in_chans=in_chans, dim=dim[i], out_dim=dim[i]*2,
                            depth=depth[i], num_heads=num_heads[i], num_inner_head=num_inner_head[i], mlp_ratio=mlp_ratio, qkv_bias=False, 
                            drop_rate=drop_rate, attn_drop_rate=attn_drop_rate, drop_path_rate=drop_path_rate, norm_layer=nn.LayerNorm))
                num_patches = self.stage[i].num_patches
                row_pixel = self.stage[i].row_pixel
                patch_pixel = self.stage[i].patch_pixel
                self.row_pos = nn.Parameter(torch.zeros(1, num_patches, row_pixel * dim[i]))
                self.column_pos = nn.Parameter(torch.zeros(1, num_patches, row_pixel * dim[i]))
                self.pixel_pos = nn.Parameter(torch.zeros(1, dim[i]*patch_pixel, num_patches * num_patches))
            
                self.downsample.append(nn.Sequential(
                            nn.Conv2d(in_chans, in_chans*2, 3, stride=2, padding=1),
                            nn.BatchNorm2d(in_chans*2),
                            nn.ReLU(inplace=True),
                            nn.Conv2d(in_chans*2, in_chans*4, 3, stride=2, padding=1),
                            nn.BatchNorm2d(in_chans*4),
                            nn.ReLU(inplace=True),
                            nn.Conv2d(in_chans*4, dim[i], 3, stride=1, padding=1),
                            nn.BatchNorm2d(dim[i]),
                            nn.ReLU(inplace=True),
                        ))
            else:
                self.stage.append(Stage(img_size=img_size//(2**(i+2)), patch_size=patch_size[i], in_chans=dim[i], dim=dim[i], out_dim=dim[i]*2,
                            depth=depth[i], num_heads=num_heads[i], num_inner_head=num_inner_head[i], mlp_ratio=mlp_ratio, qkv_bias=False, 
                            drop_rate=drop_rate, attn_drop_rate=attn_drop_rate, drop_path_rate=drop_path_rate, norm_layer=nn.LayerNorm))
                self.downsample.append(nn.AvgPool2d(kernel_size=stride[i]))

        self.norm = norm_layer(dim[3]*2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(dim[3]*2, num_classes)

        trunc_normal_(self.row_pos, std=.02)
        trunc_normal_(self.pixel_pos, std=.02)
        trunc_normal_(self.column_pos, std=.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        if isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight)
            if isinstance(m, nn.Conv2d) and m.bias is not None:
                trunc_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward_features(self, x):
        img_ds = self.downsample[0](x)
        img_merge = self.stage[0](img_ds, self.pixel_pos, self.row_pos, self.column_pos)
        for i in range(3):
            img_ds = self.downsample[i+1](img_merge)
            img_merge = self.stage[i+1](img_ds)
        return img_merge

    def forward(self, x):
        output = self.forward_features(x)
        output_flat = self.avgpool(output).flatten(1)
        output_flat = self.norm(output_flat)
        output_flat = self.head(output_flat)
        return output_flat

In [12]:
class LungDataset(Dataset):
    def __init__(self, data_files, data_dir, transform=None):
        self.data_files = data_files
        self.data_dir = data_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.data_files)
    
    def __getitem__(self, idx):
        row = self.data_files.iloc[idx]
        img_path = os.path.join(self.data_dir, row['File'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image)
        
        return image, int(row['Disease_ID'])


In [13]:
Class_types = ['Cirrhosis', 'No Fibrosis', 'Periportal Fibrosis','Portal Fibrosis','Septal Fibrosis']
data_dir = '/kaggle/input/resized-dataset256-lung-disease-raw/Resized_Dataset'

# Build dataset
train_data = []
for defects_id, sp in enumerate(Class_types):
    class_path = os.path.join(data_dir, sp)
    if os.path.exists(class_path):
        for file in os.listdir(class_path):
            train_data.append(['{}/{}'.format(sp, file), defects_id, sp])

df = pd.DataFrame(train_data, columns=['File', 'Disease_ID','Disease_Type'])
print(f"Dataset size: {len(df)}")
print(df['Disease_Type'].value_counts())

Dataset size: 6343
Disease_Type
No Fibrosis            2134
Cirrhosis              1698
Portal Fibrosis         861
Septal Fibrosis         857
Periportal Fibrosis     793
Name: count, dtype: int64


In [14]:
train_files, val_files = train_test_split(df, test_size=0.3, random_state=42, stratify=df['Disease_ID'])

In [15]:
transform_train = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats work better
])

transform_val = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [16]:
train_dataset = LungDataset(train_files, data_dir, transform=transform_train)
val_dataset = LungDataset(val_files, data_dir, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)  
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)


In [17]:
def create_lung_model():
    model = HoverTrans(
        img_size=128,
        patch_size=[16, 2, 2, 2],
        in_chans=3,
        num_classes=5,
        dim=[32, 64, 128, 256],
        depth=[1, 1, 2, 1],
        num_heads=[2, 4, 8, 16],
        num_inner_head=[2, 2, 2, 2],
        mlp_ratio=3.,
        drop_rate=0.1,
        attn_drop_rate=0.1,
        drop_path_rate=0.1
    )
    return model
    return model

In [18]:
def train_model(model, train_loader, val_loader, num_epochs=1, device='cuda'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    best_val_acc = 0.0
    train_losses, val_losses, val_accuracies = [], [], []
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
    
            # Use autocast for mixed precision
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
    
            # Scale the loss, backpropagate, and unscale the gradients
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
    
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        # Calculate metrics
        train_acc = 100. * train_correct / train_total
        val_acc = 100. * val_correct / val_total
        avg_train_loss = running_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_acc)
        
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '/kaggle/working/best_hovertrans_lung_model.pth')
            print(f'New best model saved! Val Acc: {val_acc:.2f}%')
        
        scheduler.step()
        print('-' * 60)
    
    return model, train_losses, val_losses, val_accuracies

In [ ]:
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Create model
    model = create_lung_model()
    print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

Using device: cuda


In [ ]:
def evaluate_model(model, test_loader, device='cuda', class_names=None):
    """Evaluate model performance with detailed metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    from sklearn.metrics import classification_report, confusion_matrix
    import numpy as np
    
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"Overall Accuracy: {accuracy:.4f}")
    
    if class_names:
        print("\nClassification Report:")
        print(classification_report(all_labels, all_preds, target_names=class_names))
        
        print("\nConfusion Matrix:")
        cm = confusion_matrix(all_labels, all_preds)
        print(cm)
    
    return accuracy, all_preds, all_labels


In [ ]:
def load_and_predict(model_path, image_path, transform, class_names, device='cuda'):
    """Load model and make prediction on single image"""
    model = create_lung_model()
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_class].item()
    
    return class_names[predicted_class], confidence, probabilities[0].cpu().numpy()
